## Introducción y Objetivos

La **retinopatía diabética** (RD) es una complicación ocular de la diabetes mellitus que puede provocar pérdida de visión si no se detecta a tiempo. El cribado masivo mediante fotografías del fondo de ojo (*retinografías*) permite una detección temprana, pero la revisión manual de miles de imágenes por oftalmólogos es inviable a gran escala.

Este proyecto aplica los conceptos estudiados en la asignatura para construir un **pipeline de clasificación automática** de la severidad de la RD en cinco grados:

| Grado | Nombre | Descripción clínica |
|-------|--------|---------------------|
| 0 | Sin RD | Fondo de ojo normal, sin lesiones |
| 1 | Leve | Microaneurismas aislados |
| 2 | Moderada | Hemorragias y exudados duros |
| 3 | Severa | Múltiples hemorragias, anomalías venosas |
| 4 | Proliferativa | Neovascularización, hemorragia vítrea |

### Conceptos del curso aplicados

- **Práctica 01 / Tema 1**: Naturaleza cuantitativa de la imagen médica, normalización de intensidades, importancia del preprocesamiento previo al análisis.
- **Práctica 02 / Tema 2**: Contraste local (SNR/CNR), efecto del preprocesamiento sobre la calidad de imagen, visualización de distribuciones de intensidad.
- **Práctica 04 / Tema 4**: Framework MONAI, arquitecturas de redes neuronales para imagen médica, pipeline de entrenamiento supervisado con DataLoaders y Transforms.
- **Práctica 04 / Tema 5**: Interpretabilidad mediante Grad-CAM para verificar la coherencia clínica de las predicciones.

### Dataset APTOS 2019

El dataset contiene retinografías etiquetadas con el grado de severidad de la RD, divididas en conjuntos de entrenamiento, validación y test. Presenta un **desbalance de clases** notable que habrá que tener en cuenta en el diseño del pipeline.

In [ ]:
import subprocess, sys

pkgs = [
    "monai[einops]",
    "grad-cam",
    "timm",
    "scikit-learn",
    "opencv-python-headless",
]

for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("Dependencias instaladas correctamente.")

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import monai
from monai.networks.nets import EfficientNetBN
from monai.transforms import (
    Compose,
    RandFlip,
    RandRotate90,
    RandAffine,
    RandGaussianNoise,
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    f1_score,
)

# ── Rutas al dataset ──────────────────────────────────────────────────────────
BASE      = Path("/kaggle/input/datasets/mariaherrerot/aptos2019")
TRAIN_IMG = BASE / "train_images" / "train_images"
VAL_IMG   = BASE / "val_images"   / "val_images"
TEST_IMG  = BASE / "test_images"  / "test_images"
TRAIN_CSV = BASE / "train_1.csv"
VAL_CSV   = BASE / "valid.csv"
TEST_CSV  = BASE / "test.csv"

# ── Directorio de salida para imágenes preprocesadas ─────────────────────────
PREP_TRAIN = Path("/kaggle/working/preprocessed/train")
PREP_VAL   = Path("/kaggle/working/preprocessed/val")
PREP_TEST  = Path("/kaggle/working/preprocessed/test")
for p in [PREP_TRAIN, PREP_VAL, PREP_TEST]:
    p.mkdir(parents=True, exist_ok=True)

# ── Hiperparámetros ────────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 20
LR          = 1e-4
NUM_CLASSES = 5
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_NAMES = {
    0: "Sin RD",
    1: "Leve",
    2: "Moderada",
    3: "Severa",
    4: "Proliferativa",
}

print(f"Dispositivo      : {DEVICE}")
print(f"MONAI version    : {monai.__version__}")
print(f"PyTorch version  : {torch.__version__}")

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 1. Exploración del Dataset (EDA)

Antes de construir ningún modelo, es imprescindible explorar los datos para entender su estructura y distribución. Como se aprendió en la **Práctica 01 (Tema 1)**, el primer paso en cualquier análisis de imagen médica es familiarizarse con los datos: número de muestras, distribución de clases, aspecto visual y posibles problemas.

En este caso identificamos dos aspectos clave:

1. **Desbalance de clases**: la clase 0 (Sin RD) es ampliamente mayoritaria. Este desequilibrio influirá en el diseño de la función de pérdida.
2. **Variabilidad visual**: las retinografías presentan diferencias de iluminación, tamaño y campo de visión entre imágenes, lo que motiva el preprocesamiento.

In [ ]:
# ── Lectura de CSVs ───────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train : {len(train_df)} imágenes")
print(f"Val   : {len(val_df)} imágenes")
print(f"Test  : {len(test_df)} imágenes")

# ── Distribución de clases ─────────────────────────────────────────────────────
print("\nDistribución de clases (Train):")
dist = train_df["diagnosis"].value_counts().sort_index()
for grade, count in dist.items():
    pct = count / len(train_df) * 100
    print(f"  Grado {grade} ({LABEL_NAMES[grade]:<15}): {count:>4}  ({pct:.1f}%)")

In [ ]:
# ── Gráfico de distribución de clases ────────────────────────────────────────
counts_train = train_df["diagnosis"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(
    [f"G{i}\n{LABEL_NAMES[i]}" for i in counts_train.index],
    counts_train.values,
    color=["#4CAF50", "#FFC107", "#FF9800", "#F44336", "#9C27B0"],
)
for bar, val in zip(bars, counts_train.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        str(val),
        ha="center",
        fontsize=10,
    )
ax.set_title("Distribución de clases en el conjunto de entrenamiento", fontsize=13)
ax.set_ylabel("Número de imágenes")
ax.set_ylim(0, counts_train.max() * 1.18)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ── Una imagen por grado ──────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(1, 5, figsize=(18, 4))
for grade in range(5):
    sample_id = train_df[train_df["diagnosis"] == grade]["id_code"].iloc[0]
    img = mpimg.imread(str(TRAIN_IMG / f"{sample_id}.png"))
    ax2[grade].imshow(img)
    ax2[grade].set_title(f"Grado {grade}\n{LABEL_NAMES[grade]}", fontsize=10)
    ax2[grade].axis("off")
fig2.suptitle("Muestra de imagen por grado de severidad", fontsize=13, y=1.02)
fig2.tight_layout()
plt.show()

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 2. Preprocesamiento Específico de Imagen de Fondo de Ojo

### Motivación

Las retinografías comparten con las imágenes médicas estudiadas en prácticas un principio fundamental: requieren un **preprocesamiento específico** antes de cualquier análisis cuantitativo. Como se vio en la **Práctica 01 (Tema 1)**, la imagen médica no puede analizarse «tal cual» sin adaptar primero su escala e intensidades.

En la **Práctica 02 (Tema 2)** también se estudió cómo el contraste local (medido con SNR/CNR) afecta directamente a la detectabilidad de estructuras patológicas. En retinografías, los microaneurismas y exudados duros son lesiones de pequeño tamaño y bajo contraste local que pueden perderse si no se mejora el contraste de la imagen.

### Etapa 1 — Máscara circular

Las retinografías presentan un **fondo negro** fuera del disco retiniano que no contiene información clínica. Este artefacto de adquisición puede confundir al modelo, ya que aprende características irrelevantes del borde negro. Se elimina mediante una máscara circular basada en el canal verde (el de mayor contraste en retinografías).

### Etapa 2 — CLAHE (Contrast Limited Adaptive Histogram Equalization)

CLAHE mejora el contraste de forma **adaptativa y localizada**, operando sobre el canal L del espacio de color LAB. A diferencia de la ecualización global de histograma (que puede amplificar ruido), CLAHE trabaja sobre pequeñas regiones (*tiles*) y limita la amplificación (*clip limit*), preservando la estructura global de la imagen. Esto conecta directamente con los conceptos de contraste local estudiados en la **Práctica 02**.

### Normalización

Tras el preprocesamiento, las imágenes se normalizan con la media y desviación estándar de ImageNet, ya que se usará *transfer learning* desde pesos preentrenados en ese conjunto. Esta normalización sigue el principio visto en la **Práctica 01**: los valores de píxel deben llevarse a un rango comparable antes de cualquier análisis.

In [ ]:
def apply_circular_mask(img: np.ndarray, tol: int = 7) -> np.ndarray:
    '''
    Aplica una máscara circular para eliminar el fondo negro fuera del disco
    retiniano. Trabaja sobre el canal verde (índice 1), que es el de mayor
    contraste en imágenes de fondo de ojo.

    Args:
        img : imagen RGB en uint8 (H, W, 3).
        tol : umbral de intensidad para considerar un píxel como fondo.
    Returns:
        Imagen recortada al área retiniana con el fondo negro a cero.
    '''
    gray = img[:, :, 1]  # canal verde
    mask = gray > tol

    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any() or not cols.any():
        return img

    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]
    img_cropped = img[r0 : r1 + 1, c0 : c1 + 1]

    h, w = img_cropped.shape[:2]
    cy, cx = h // 2, w // 2
    radius = min(cx, cy)
    Y, X = np.ogrid[:h, :w]
    circle_mask = (X - cx) ** 2 + (Y - cy) ** 2 <= radius ** 2

    result = img_cropped.copy()
    result[~circle_mask] = 0
    return result


def apply_clahe(
    img: np.ndarray, clip_limit: float = 2.0, tile_grid: tuple = (8, 8)
) -> np.ndarray:
    '''
    Aplica CLAHE sobre el canal L del espacio LAB.
    Mejora el contraste local de forma adaptativa sin amplificar el ruido,
    lo que resalta microaneurismas y exudados de bajo contraste.

    Args:
        img        : imagen RGB en uint8.
        clip_limit : límite de amplificación del contraste.
        tile_grid  : tamaño de la rejilla de tiles (adaptación local).
    Returns:
        Imagen con contraste local mejorado en uint8.
    '''
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    lab[:, :, 0] = clahe_obj.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def preprocess_image(img_path: str, size: int = IMG_SIZE) -> np.ndarray:
    '''
    Pipeline completo de preprocesamiento:
      1. Máscara circular  (elimina fondo negro).
      2. CLAHE             (mejora contraste local).
      3. Redimensionado    (size × size).
      4. Normalización     (float32, rango [0, 1]).

    Args:
        img_path : ruta al archivo PNG.
        size     : tamaño de salida cuadrado en píxeles.
    Returns:
        Array float32 de shape (size, size, 3) con valores en [0, 1].
    '''
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = apply_circular_mask(img)
    img = apply_clahe(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0


print("Funciones de preprocesamiento definidas.")

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(12, 20))
col_titles = ["Original (redimensionada)", "Tras máscara circular", "Pipeline completo (+ CLAHE)"]

for row, grade in enumerate(range(5)):
    sample_id = train_df[train_df["diagnosis"] == grade]["id_code"].iloc[0]
    path = TRAIN_IMG / f"{sample_id}.png"

    img_raw = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

    # Etapa 1: solo máscara circular
    stage1 = apply_circular_mask(img_raw)
    stage1_r = cv2.resize(stage1, (IMG_SIZE, IMG_SIZE))

    # Etapas completas
    final = (preprocess_image(path) * 255).astype(np.uint8)

    img_raw_r = cv2.resize(img_raw, (IMG_SIZE, IMG_SIZE))

    for col, (stage_img, title) in enumerate(
        zip([img_raw_r, stage1_r, final], col_titles)
    ):
        ax = axes[row][col]
        ax.imshow(stage_img)
        ax.axis("off")
        if row == 0:
            ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
        if col == 0:
            ax.set_ylabel(
                f"Grado {grade}\n{LABEL_NAMES[grade]}",
                fontsize=11,
                rotation=0,
                labelpad=75,
                va="center",
            )

fig.suptitle(
    "Efecto del preprocesamiento por grado de severidad",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
def preprocess_and_save(df: pd.DataFrame, src_dir: Path, dst_dir: Path, desc: str = ""):
    '''
    Aplica el pipeline de preprocesamiento a todas las imágenes de un split
    y las guarda en dst_dir como PNG uint8.
    Guardar en disco evita reprocesar en cada época, acelerando el entrenamiento.
    '''
    for _, row in df.iterrows():
        src = src_dir / f"{row['id_code']}.png"
        dst = dst_dir / f"{row['id_code']}.png"
        if dst.exists():
            continue
        img = (preprocess_image(src) * 255).astype(np.uint8)
        cv2.imwrite(str(dst), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    print(f"  [{desc}] {len(df)} imágenes guardadas en {dst_dir}")


print("Preprocesando y guardando imágenes en disco...")
preprocess_and_save(train_df, TRAIN_IMG, PREP_TRAIN, "Train")
preprocess_and_save(val_df,   VAL_IMG,   PREP_VAL,   "Val")
preprocess_and_save(test_df,  TEST_IMG,  PREP_TEST,  "Test")
print("¡Listo!")

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 3. Dataset y DataLoaders con MONAI

Como se estudió en la **Práctica 04 (Tema 4)**, el framework **MONAI** proporciona herramientas especializadas para construir pipelines de datos en imagen médica. En esta sección adaptamos sus componentes a las retinografías.

### Transforms de MONAI (aumentación de datos)

Las **transforms** de MONAI permiten aplicar aumentaciones aleatorias durante el entrenamiento para mejorar la capacidad de generalización del modelo. Son las mismas que se usaron en la Práctica 04 con DynUNet:

- `RandFlip`: volteo horizontal/vertical aleatorio.
- `RandRotate90`: rotaciones en múltiplos de 90°.
- `RandAffine`: deformaciones afines leves (escala, rotación suave).
- `RandGaussianNoise`: ruido gaussiano ligero para regularización.

Durante la **validación y el test** no se aplica ninguna aumentación, para que la evaluación refleje fielmente el rendimiento del modelo.

### Manejo del desbalance de clases

El dataset APTOS 2019 está claramente desbalanceado. Una solución sencilla y directa es usar una **`CrossEntropyLoss` ponderada**: se asigna a cada clase un peso inversamente proporcional a su frecuencia, de modo que el modelo penaliza más los errores en clases minoritarias (Grados 3 y 4). Este enfoque no requiere modificar el DataLoader y es suficiente para este caso.

In [ ]:
# ── Estadísticas de normalización ImageNet ────────────────────────────────────
# Se usan para compatibilidad con los pesos preentrenados en ImageNet
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class RetinopathyDataset(Dataset):
    '''
    Dataset de retinografías compatible con PyTorch y MONAI.
    Carga las imágenes preprocesadas, aplica normalización ImageNet
    y convierte a tensor (C, H, W).
    '''

    def __init__(
        self,
        df: pd.DataFrame,
        img_dir: Path,
        transforms=None,
    ):
        self.df         = df.reset_index(drop=True)
        self.img_dir    = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = self.img_dir / f"{row['id_code']}.png"

        # Cargar imagen preprocesada (ya en disco) y normalizar a [0,1]
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0

        # Normalización por canal (ImageNet: compatible con transfer learning)
        img = (img - MEAN) / STD

        # (H, W, C) → (C, H, W) para PyTorch
        img = torch.from_numpy(img.transpose(2, 0, 1)).float()

        if self.transforms:
            img = self.transforms(img)

        return img, int(row["diagnosis"])


print("Clase RetinopathyDataset definida.")

In [ ]:
# ── Transforms MONAI ──────────────────────────────────────────────────────────
train_transforms = Compose([
    RandFlip(prob=0.5, spatial_axis=1),                                  # flip horizontal
    RandFlip(prob=0.5, spatial_axis=0),                                  # flip vertical
    RandRotate90(prob=0.5),                                              # rotación 90°
    RandAffine(prob=0.3, rotate_range=0.2, scale_range=0.1,             # afín leve
               padding_mode="zeros"),
    RandGaussianNoise(prob=0.2, mean=0.0, std=0.05),                    # ruido gaussiano
])

# Sin aumentación en validación y test (evaluación fiel del rendimiento)
eval_transforms = None

# ── Datasets ───────────────────────────────────────────────────────────────────
train_ds = RetinopathyDataset(train_df, PREP_TRAIN, train_transforms)
val_ds   = RetinopathyDataset(val_df,   PREP_VAL,   eval_transforms)
test_ds  = RetinopathyDataset(test_df,  PREP_TEST,  eval_transforms)

# ── DataLoaders ────────────────────────────────────────────────────────────────
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

print(f"Batches por época (train) : {len(train_loader)}")
print(f"Batches de validación     : {len(val_loader)}")
print(f"Batches de test           : {len(test_loader)}")

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 4. Modelo: EfficientNetBN con MONAI

Como se estudió en la **Práctica 04 (Tema 4)**, el framework MONAI incluye en `monai.networks.nets` una colección de arquitecturas de red neuronal para imagen médica. En esta práctica usamos **EfficientNet-B3**, disponible como `EfficientNetBN`.

### Transfer learning desde ImageNet

Se usa `pretrained=True` para inicializar los pesos del backbone desde ImageNet. El *transfer learning* es especialmente útil cuando el dataset es de tamaño moderado (como APTOS 2019): las capas convolucionales ya han aprendido detectores de bordes, texturas y formas que son útiles también para retinografías médicas.

### Pesos de clase para CrossEntropyLoss

Para abordar el desbalance de clases calculamos un tensor de pesos inversamente proporcional a la frecuencia de cada clase. La `CrossEntropyLoss` de PyTorch acepta este tensor directamente, de modo que los errores en clases minoritarias (Grado 3 — Severa, Grado 4 — Proliferativa) reciben una penalización mayor durante el entrenamiento.

In [ ]:
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    '''Construye EfficientNet-B3 desde MONAI con pesos ImageNet preentrenados.'''
    model = EfficientNetBN(
        model_name="efficientnet-b3",
        pretrained=True,
        spatial_dims=2,
        in_channels=3,
        num_classes=num_classes,
    )
    return model.to(DEVICE)


def build_criterion(df: pd.DataFrame) -> nn.CrossEntropyLoss:
    '''
    Calcula pesos de clase inversamente proporcionales a su frecuencia y
    devuelve CrossEntropyLoss ponderada.
    '''
    counts  = df["diagnosis"].value_counts().sort_index().values.astype(float)
    weights = (1.0 / counts) / (1.0 / counts).sum() * NUM_CLASSES
    return nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    )


model     = build_model()
criterion = build_criterion(train_df)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

n_params = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B3 parámetros totales: {n_params:,}")

# Pesos de clase calculados
counts_np = train_df["diagnosis"].value_counts().sort_index().values.astype(float)
w = (1.0 / counts_np) / (1.0 / counts_np).sum() * NUM_CLASSES
print("\nPesos de clase para CrossEntropyLoss:")
for i, wi in enumerate(w):
    print(f"  Grado {i} ({LABEL_NAMES[i]:<15}): {wi:.4f}")

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 5. Entrenamiento

El bucle de entrenamiento sigue el mismo esquema visto en la **Práctica 04 (Tema 4)** con DynUNet para segmentación:

1. En cada época se recorre el `train_loader` calculando la pérdida y actualizando los pesos con Adam.
2. Al finalizar cada época, se evalúa sobre el `val_loader`.
3. Se guarda el **mejor checkpoint** según el **kappa cuadrático ponderado** de validación.

### Kappa cuadrático ponderado

La métrica principal para este dataset es el **kappa cuadrático ponderado** (función `cohen_kappa_score` con `weights="quadratic"`). Esta métrica es el estándar para tareas de clasificación ordinal en oftalmología, ya que penaliza los errores de forma proporcional al cuadrado de la distancia entre la clase predicha y la real: equivocarse entre Grado 0 y Grado 4 penaliza mucho más que equivocarse entre Grado 0 y Grado 1.

> Es la métrica oficial de la competición APTOS 2019 y el estándar en la literatura de gradación automática de retinopatía.

In [ ]:
import copy


def train_one_epoch(model, loader, optimizer, criterion):
    '''Entrena el modelo durante una época. Devuelve la pérdida media.'''
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    '''
    Evalúa el modelo sobre un loader.
    Devuelve (predicciones, etiquetas_reales) como arrays NumPy.
    '''
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


def quadratic_kappa(preds, labels):
    '''Kappa cuadrático ponderado: métrica principal del dataset APTOS 2019.'''
    return cohen_kappa_score(labels, preds, weights="quadratic")


print("Funciones de entrenamiento y evaluación definidas.")

In [ ]:
# ── Checkpoint de caché ───────────────────────────────────────────────────────
# Si el archivo ya existe en /kaggle/working, se carga directamente y se omite
# el bucle de entrenamiento, ahorrando GPU y tiempo en ejecuciones posteriores.
# Para forzar un reentrenamiento basta con borrar el archivo .pth.

CKPT_PATH = Path("/kaggle/working/best_model.pth")

if CKPT_PATH.exists():
    print(f"[CACHE] Checkpoint encontrado: {CKPT_PATH}")
    print("        Cargando pesos y omitiendo entrenamiento...")
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    model.eval()
    # Recuperamos el kappa de validación para que las celdas siguientes funcionen
    val_preds_ck, val_labels_ck = evaluate(model, val_loader)
    best_kappa = quadratic_kappa(val_preds_ck, val_labels_ck)
    best_epoch = -1   # desconocido al cargar desde caché
    history    = {"train_loss": [], "val_kappa": [best_kappa]}
    print(f"        Kappa en validación (checkpoint): {best_kappa:.4f}")

else:
    history    = {"train_loss": [], "val_kappa": []}
    best_kappa = -1.0
    best_epoch = -1
    best_state = None

    print(f"Entrenando EfficientNet-B3 durante {NUM_EPOCHS} épocas...")
    print("=" * 65)

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss            = train_one_epoch(model, train_loader, optimizer, criterion)
        val_preds, val_labels = evaluate(model, val_loader)
        val_kappa             = quadratic_kappa(val_preds, val_labels)

        history["train_loss"].append(train_loss)
        history["val_kappa"].append(val_kappa)

        tag = ""
        if val_kappa > best_kappa:
            best_kappa = val_kappa
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save(best_state, CKPT_PATH)
            tag = "  ← mejor"

        print(
            f"Época {epoch:02d}/{NUM_EPOCHS}  |  "
            f"Loss train: {train_loss:.4f}  |  "
            f"Val kappa: {val_kappa:.4f}{tag}"
        )

    print(f"\nMejor kappa de validación: {best_kappa:.4f}  (época {best_epoch})")
    print(f"Checkpoint guardado en: {CKPT_PATH}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(history["train_loss"]) > 0:
    # Entrenamiento realizado en esta sesión → curvas completas
    epochs_x = list(range(1, len(history["train_loss"]) + 1))

    axes[0].plot(epochs_x, history["train_loss"], color="#667eea", linewidth=2)
    axes[0].set_title("Pérdida de entrenamiento", fontweight="bold")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("CrossEntropy Loss (ponderada)")
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_x, history["val_kappa"], color="#667eea", linewidth=2,
                 marker="o", markersize=4)
    best_ep_idx = history["val_kappa"].index(max(history["val_kappa"]))
    axes[1].scatter(
        best_ep_idx + 1, history["val_kappa"][best_ep_idx],
        color="red", s=80, zorder=5,
        label=f"Mejor (κ={history['val_kappa'][best_ep_idx]:.4f}, época {best_ep_idx+1})",
    )
    axes[1].legend()

else:
    # Modelo cargado desde caché → no hay historial de épocas
    axes[0].text(0.5, 0.5, "Modelo cargado desde caché\n(sin historial de entrenamiento)",
                 ha="center", va="center", fontsize=12, color="gray",
                 transform=axes[0].transAxes)
    axes[1].axhline(best_kappa, color="#667eea", linewidth=2, linestyle="--",
                    label=f"Kappa en validación: {best_kappa:.4f}")
    axes[1].legend()

axes[1].set_title("Kappa cuadrático ponderado — Validación", fontweight="bold")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Kappa")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

plt.suptitle("Curvas de entrenamiento — EfficientNet-B3 (MONAI)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 6. Evaluación Final en el Conjunto de Test

Se carga el mejor checkpoint (guardado según el kappa de validación) y se evalúa sobre el **conjunto de test**, que no ha participado en ninguna decisión de entrenamiento ni de selección de modelo.

Se reportan:
- **Kappa cuadrático ponderado**: métrica principal, penaliza errores ordinalmente.
- **F1-score por clase**: permite identificar qué grados se clasifican mejor o peor, especialmente relevante en presencia de desbalance.
- **Matriz de confusión normalizada**: revela qué grados se confunden entre sí con mayor frecuencia.

In [ ]:
# ── Cargar el mejor checkpoint ────────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

test_preds, test_labels = evaluate(model, test_loader)
test_kappa = quadratic_kappa(test_preds, test_labels)
test_f1    = f1_score(test_labels, test_preds, average="macro")

print("=" * 55)
print("RESULTADOS EN TEST")
print("=" * 55)
print(f"  Kappa cuadrático ponderado : {test_kappa:.4f}")
print(f"  F1-score macro             : {test_f1:.4f}")
print()
print(classification_report(
    test_labels,
    test_preds,
    target_names=[LABEL_NAMES[i] for i in range(5)],
))

In [ ]:
cm = confusion_matrix(test_labels, test_preds, normalize="true")
class_names = [f"G{i}\n{LABEL_NAMES[i]}" for i in range(5)]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(5));  ax.set_xticklabels(class_names, fontsize=9)
ax.set_yticks(range(5));  ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel("Predicción")
ax.set_ylabel("Etiqueta real")
ax.set_title(
    f"Matriz de confusión (Test) — Kappa = {test_kappa:.3f}",
    fontsize=12,
    fontweight="bold",
)

for i in range(5):
    for j in range(5):
        ax.text(
            j, i,
            f"{cm[i, j]:.2f}",
            ha="center", va="center",
            color="white" if cm[i, j] > 0.5 else "black",
            fontsize=9,
        )

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# ── Resumen numérico ───────────────────────────────────────────────────────────
print("\nF1-score por clase:")
f1_per = f1_score(test_labels, test_preds, average=None)
for i, f1_i in enumerate(f1_per):
    print(f"  Grado {i} ({LABEL_NAMES[i]:<15}): {f1_i:.4f}")

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 7. Interpretabilidad: Mapas Grad-CAM

Como se estudió en la **Práctica 04 (Tema 4)** y el **Tema 5** de la asignatura (*Interpretabilidad y fiabilidad de la IA en el ámbito clínico*), verificar que el modelo basa sus decisiones en las regiones clínicamente relevantes es fundamental antes de cualquier aplicación médica.

**Grad-CAM** calcula el gradiente de la puntuación de la clase predicha respecto a los mapas de activación de la **última capa convolucional**. Las regiones con gradientes altos son las que más influyen en la decisión del modelo. Para las retinografías:

- **Grado 0** (Sin RD): se esperan activaciones difusas, sin un foco concreto.
- **Grados 1-4**: se esperan activaciones concentradas en la **región macular** y los vasos sanguíneos periféricos, donde se localizan los microaneurismas, hemorragias y exudados.

Si los mapas Grad-CAM muestran activaciones en estas regiones anatómicas, podemos confiar en que el modelo está aprendiendo características clínicamente relevantes y **no artefactos** del preprocesamiento o el fondo negro.

> La misma capa objetivo (`_conv_head`) que se usó en la Práctica 04 con EfficientNet es válida aquí.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Capa objetivo: última convolución de EfficientNet-B3 antes del pooling global
target_layer = [model._conv_head]
cam = GradCAM(model=model, target_layers=target_layer)

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
col_labels = ["Imagen preprocesada", "Mapa Grad-CAM", "Superposición"]

for row, grade in enumerate(range(5)):
    # Buscar una imagen del grado en el conjunto de test
    sample_id = test_df[test_df["diagnosis"] == grade]["id_code"].iloc[0]
    path_prep = PREP_TEST / f"{sample_id}.png"

    img_prep  = cv2.cvtColor(cv2.imread(str(path_prep)), cv2.COLOR_BGR2RGB)
    img_float = img_prep.astype(np.float32) / 255.0

    # Tensor normalizado para el modelo
    img_norm = (img_float - MEAN) / STD
    tensor   = torch.from_numpy(img_norm.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    # Predicción del modelo (para mostrarlo en el título)
    with torch.no_grad():
        pred_grade = int(model(tensor).argmax(dim=1).item())

    # Grad-CAM para la clase verdadera del grado
    targets       = [ClassifierOutputTarget(grade)]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]
    overlay       = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    for col, (display_img, title) in enumerate(
        zip([img_prep, (grayscale_cam * 255).astype(np.uint8), overlay], col_labels)
    ):
        ax = axes[row][col]
        cmap = "jet" if col == 1 else None
        ax.imshow(display_img, cmap=cmap)
        ax.axis("off")
        if row == 0:
            ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
        if col == 0:
            match_str = "OK" if pred_grade == grade else f"err→G{pred_grade}"
            ax.set_ylabel(
                f"Grado {grade}\n{LABEL_NAMES[grade]}\n({match_str})",
                fontsize=10,
                rotation=0,
                labelpad=90,
                va="center",
            )

fig.suptitle(
    "Mapas Grad-CAM por grado de severidad — EfficientNet-B3 (MONAI)",
    fontsize=13,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## 8. Conclusiones

Este proyecto ha aplicado los conceptos estudiados a lo largo de las cuatro prácticas de la asignatura para construir un pipeline completo de clasificación automática de retinopatía diabética:

| Sección | Concepto del curso aplicado | Práctica |
|---------|----------------------------|----------|
| Preprocesamiento (máscara circular) | Artefactos de adquisición, imagen cuantitativa | P01 / Tema 1 |
| Preprocesamiento (CLAHE) | Contraste local, SNR/CNR | P02 / Tema 2 |
| Normalización ImageNet | Normalización de intensidades | P01 / Tema 1 |
| MONAI Transforms (aumentación) | Pipeline de datos MONAI | P04 / Tema 4 |
| EfficientNetBN con transfer learning | MONAI networks, deep learning médico | P04 / Tema 4 |
| CrossEntropyLoss ponderada | Manejo del desbalance de clases | P04 / Tema 4 |
| Kappa cuadrático + F1 + conf. matrix | Evaluación de clasificadores | P04 / Tema 4 |
| Grad-CAM | Interpretabilidad en IA clínica | P04 / Tema 5 |

### Análisis de los resultados Grad-CAM

Los mapas de activación permiten verificar si el modelo localiza las zonas clínicamente relevantes:
- En los grados con lesión (1-4) se esperan activaciones en la región macular y vasos.
- En el Grado 0 las activaciones deberían ser más difusas.
- Si el modelo activa zonas del fondo negro o bordes del artefacto circular, indicaría que el preprocesamiento no ha eliminado correctamente esos artefactos.

### Limitaciones

- El conjunto de entrenamiento es de tamaño moderado; más datos mejorarían la generalización.
- La normalización de iluminación podría refinarse con técnicas más avanzadas.
- En un contexto clínico real sería necesaria una **validación externa** en datos de otro centro o equipo de adquisición.

# EXPERIMENTOS DE MEJORA???

In [ ]:
import os, copy, warnings
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import monai
from monai.networks.nets import EfficientNetBN
from monai.transforms import (
    Compose, RandFlip, RandRotate90, RandAffine, RandGaussianNoise,
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, f1_score,
)
warnings.filterwarnings("ignore")

# ── Rutas ─────────────────────────────────────────────────────────────────────
BASE      = Path("/kaggle/input/datasets/mariaherrerot/aptos2019")
TRAIN_IMG = BASE / "train_images" / "train_images"
VAL_IMG   = BASE / "val_images"   / "val_images"
TEST_IMG  = BASE / "test_images"  / "test_images"
TRAIN_CSV = BASE / "train_1.csv"
VAL_CSV   = BASE / "valid.csv"
TEST_CSV  = BASE / "test.csv"
PREP_TRAIN = Path("/kaggle/working/preprocessed/train")
PREP_VAL   = Path("/kaggle/working/preprocessed/val")
PREP_TEST  = Path("/kaggle/working/preprocessed/test")

# ── Hiperparámetros ───────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 20
LR          = 1e-4
NUM_CLASSES = 5
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
LABEL_NAMES = {0: "Sin RD", 1: "Leve", 2: "Moderada", 3: "Severa", 4: "Proliferativa"}

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)


# ── Funciones de preprocesamiento (del notebook baseline) ────────────────────
def apply_circular_mask(img, tol=7):
    gray = img[:, :, 1]
    mask = gray > tol
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any() or not cols.any():
        return img
    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]
    img_c = img[r0:r1+1, c0:c1+1]
    h, w = img_c.shape[:2]
    cy, cx = h//2, w//2
    radius = min(cx, cy)
    Y, X = np.ogrid[:h, :w]
    circle_mask = (X-cx)**2 + (Y-cy)**2 <= radius**2
    result = img_c.copy()
    result[~circle_mask] = 0
    return result

def apply_clahe(img, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    lab[:, :, 0] = clahe_obj.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def preprocess_image(img_path, size=IMG_SIZE):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = apply_circular_mask(img)
    img = apply_clahe(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0


# ── Dataset base ──────────────────────────────────────────────────────────────
class RetinopathyDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = self.img_dir / f"{row['id_code']}.png"
        img  = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        img  = img.astype(np.float32) / 255.0
        img  = (img - MEAN) / STD
        img  = torch.from_numpy(img.transpose(2, 0, 1)).float()
        if self.transforms:
            img = self.transforms(img)
        return img, int(row["diagnosis"])


# ── Métricas ──────────────────────────────────────────────────────────────────
def quadratic_kappa(preds, labels):
    return cohen_kappa_score(labels, preds, weights="quadratic")

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


# ── Construcción de modelo ────────────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    model = EfficientNetBN(
        model_name="efficientnet-b3",
        pretrained=True,
        spatial_dims=2,
        in_channels=3,
        num_classes=num_classes,
    )
    return model.to(DEVICE)

def build_criterion(df, n_classes=NUM_CLASSES):
    counts  = df["diagnosis"].value_counts().sort_index().values.astype(float)
    weights = (1.0 / counts) / (1.0 / counts).sum() * n_classes
    return nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    )

print(f"Configuración cargada. Dispositivo: {DEVICE}")

In [ ]:
# Resultado del modelo baseline (del notebook anterior, cópialo aquí si lo tienes)
# Si acabas de terminar el baseline, déjalo en esta variable para comparar.
BASELINE_KAPPA = test_kappa
BASELINE_F1    = test_f1
print(f"Baseline de referencia — Kappa: {BASELINE_KAPPA:.4f} | F1 macro: {BASELINE_F1:.4f}")

Experimento A — Sobremuestreo (P02 + P04)

El más sencillo. Solo modifica el DataFrame antes de crear el Dataset, sin tocar nada más. Si G3 mejora, confirma que el desbalance era la causa principal. Conecta con P02 (análisis de distribuciones) y P04 (MONAI transforms garantizan que los duplicados no sean copias exactas).

In [ ]:
# ── A.1 — Visualización de errores del baseline ──────────────────────────────
# Cargamos el checkpoint del baseline y buscamos qué casos se predicen mal.

baseline_ckpt = Path("/kaggle/working/best_model.pth")

if not baseline_ckpt.exists():
    print("[!] No se encontró best_model.pth del baseline.")
    print("    Ejecuta primero el notebook baseline para generar el checkpoint.")
else:
    eval_transforms_A = None
    test_ds_A  = RetinopathyDataset(test_df, PREP_TEST, eval_transforms_A)
    test_loader_A = DataLoader(test_ds_A, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=2, pin_memory=True)

    model_baseline = build_model()
    model_baseline.load_state_dict(torch.load(baseline_ckpt, map_location=DEVICE))

    test_preds_base, test_labels_base = evaluate(model_baseline, test_loader_A)
    kappa_base_check = quadratic_kappa(test_preds_base, test_labels_base)
    print(f"Kappa baseline (verificación): {kappa_base_check:.4f}")

    # Índices de error por clase
    error_mask = test_preds_base != test_labels_base
    print(f"Errores totales: {error_mask.sum()} / {len(test_labels_base)} "
          f"({100*error_mask.mean():.1f}%)")
    print()
    print("Errores por clase verdadera:")
    for g in range(5):
        mask_g = test_labels_base == g
        err_g  = (test_preds_base[mask_g] != g).sum()
        total_g = mask_g.sum()
        print(f"  G{g} {LABEL_NAMES[g]:<15}: "
              f"{err_g}/{total_g} errores ({100*err_g/max(total_g,1):.0f}%)")

In [ ]:
if baseline_ckpt.exists():
    # ── Matriz de confusión del baseline (detallada) ──────────────────────────
    cm = confusion_matrix(test_labels_base, test_preds_base, normalize="true")
    class_names = [f"G{i} {LABEL_NAMES[i]}" for i in range(5)]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(5)); ax.set_xticklabels(class_names, rotation=30,
                                                  ha="right", fontsize=9)
    ax.set_yticks(range(5)); ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real")
    ax.set_title("Baseline — Matriz de confusión normalizada", fontweight="bold")

    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i,j] > 0.55 else "black", fontsize=9)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    # Observación: qué grados se confunden más entre sí
    print("\nObservaciones clave de la matriz de confusión:")
    for i in range(1, 5):  # solo clases con RD
        worst_pred = cm[i].argmax()
        if worst_pred != i:
            print(f"  G{i} ({LABEL_NAMES[i]}) se confunde principalmente "
                  f"con G{worst_pred} ({LABEL_NAMES[worst_pred]}) — "
                  f"{cm[i,worst_pred]:.2f} de los casos)")

In [ ]:
# ── A.2 — Sobremuestreo manual de clases minoritarias ────────────────────────
# Objetivo: equilibrar la distribución sin modificar el DataLoader.
# Repetimos filas del DataFrame de entrenamiento para G1, G3 y G4
# hasta que su frecuencia sea al menos el 12% del dataset total.

TARGET_FREQ = 0.12  # proporción mínima deseada para cada clase

def oversample_df(df, target_freq=TARGET_FREQ, seed=42):
    '''
    Sobremuestrea las clases minoritarias repitiendo sus filas hasta
    que cada clase represente al menos `target_freq` del total.
    Itera solo sobre los grados presentes en el DataFrame, por lo que
    funciona tanto con los 5 grados completos como con subconjuntos
    (por ejemplo, solo G1-G4 reindexados a 0-3 en la Etapa 2).
    '''
    rng       = np.random.RandomState(seed)
    df_result = df.copy()
    total     = len(df)

    for grade in sorted(df["diagnosis"].unique()):
        n_current  = (df_result["diagnosis"] == grade).sum()
        n_target   = int(target_freq * (total + n_current))
        if n_current >= n_target:
            continue
        n_extra    = n_target - n_current
        grade_rows = df[df["diagnosis"] == grade]
        extra_rows = grade_rows.sample(n=n_extra, replace=True, random_state=rng)
        df_result  = pd.concat([df_result, extra_rows], ignore_index=True)
        print(f"  G{grade} ({LABEL_NAMES.get(grade, str(grade)):<15}): "
              f"{n_current} → {n_current + n_extra} muestras (+{n_extra})")

    return df_result.sample(frac=1, random_state=seed).reset_index(drop=True)


print("Distribución original (Train):")
for g in range(5):
    n = (train_df["diagnosis"] == g).sum()
    pct = 100 * n / len(train_df)
    print(f"  G{g} ({LABEL_NAMES[g]:<15}): {n} ({pct:.1f}%)")

print()
print("Aplicando sobremuestreo...")
train_df_os = oversample_df(train_df, target_freq=TARGET_FREQ)

print()
print(f"Distribución tras sobremuestreo ({len(train_df_os)} muestras):")
for g in range(5):
    n = (train_df_os["diagnosis"] == g).sum()
    pct = 100 * n / len(train_df_os)
    print(f"  G{g} ({LABEL_NAMES[g]:<15}): {n} ({pct:.1f}%)")

In [ ]:
# ── A.3 — Entrenamiento con sobremuestreo ─────────────────────────────────────

train_transforms_A = Compose([
    RandFlip(prob=0.5, spatial_axis=1),
    RandFlip(prob=0.5, spatial_axis=0),
    RandRotate90(prob=0.5),
    RandAffine(prob=0.3, rotate_range=0.2, scale_range=0.1,
               padding_mode="zeros"),
    RandGaussianNoise(prob=0.2, mean=0.0, std=0.05),
])

train_ds_A = RetinopathyDataset(train_df_os, PREP_TRAIN, train_transforms_A)
val_ds_A   = RetinopathyDataset(val_df,      PREP_VAL,   None)
test_ds_A  = RetinopathyDataset(test_df,     PREP_TEST,  None)

loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
train_loader_A = DataLoader(train_ds_A, shuffle=True,  **loader_kw)
val_loader_A   = DataLoader(val_ds_A,   shuffle=False, **loader_kw)
test_loader_A  = DataLoader(test_ds_A,  shuffle=False, **loader_kw)

model_A     = build_model()
criterion_A = build_criterion(train_df_os)
optimizer_A = torch.optim.Adam(model_A.parameters(), lr=LR, weight_decay=1e-5)

CKPT_A = Path("/kaggle/working/best_model_A.pth")

def train_one_epoch_fn(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

# ── Checkpoint de caché ───────────────────────────────────────────────────────
if CKPT_A.exists():
    print(f"[CACHE] Checkpoint encontrado: {CKPT_A}")
    print("        Cargando pesos y omitiendo entrenamiento (Exp A)...")
    model_A.load_state_dict(torch.load(CKPT_A, map_location=DEVICE))
    model_A.eval()
    val_preds_ck_A, val_labels_ck_A = evaluate(model_A, val_loader_A)
    best_kappa_A = quadratic_kappa(val_preds_ck_A, val_labels_ck_A)
    history_A    = {"train_loss": [], "val_kappa": [best_kappa_A]}
    print(f"        Kappa en validación (checkpoint A): {best_kappa_A:.4f}")

else:
    history_A    = {"train_loss": [], "val_kappa": []}
    best_kappa_A = -1.0
    best_state_A = None

    print(f"Batches/época (train con sobremuestreo): {len(train_loader_A)}")
    print(f"Iniciando entrenamiento Exp A ({NUM_EPOCHS} épocas)...")
    print("=" * 65)

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch_fn(model_A, train_loader_A, optimizer_A, criterion_A)
        val_preds_A, val_labels_A = evaluate(model_A, val_loader_A)
        val_kappa_A = quadratic_kappa(val_preds_A, val_labels_A)

        history_A["train_loss"].append(train_loss)
        history_A["val_kappa"].append(val_kappa_A)

        tag = ""
        if val_kappa_A > best_kappa_A:
            best_kappa_A = val_kappa_A
            best_state_A = copy.deepcopy(model_A.state_dict())
            torch.save(best_state_A, CKPT_A)
            tag = "  ← mejor"

        print(f"Época {epoch:02d}/{NUM_EPOCHS}  |  "
              f"Loss: {train_loss:.4f}  |  Val kappa: {val_kappa_A:.4f}{tag}")

    print(f"\nMejor kappa validación (Exp A): {best_kappa_A:.4f}")


In [ ]:
# ── A.4 — Evaluación en test ──────────────────────────────────────────────────
model_A.load_state_dict(torch.load(CKPT_A, map_location=DEVICE))
test_preds_A, test_labels_A = evaluate(model_A, test_loader_A)

kappa_A = quadratic_kappa(test_preds_A, test_labels_A)
f1_A    = f1_score(test_labels_A, test_preds_A, average="macro")

print("=" * 55)
print("EXPERIMENTO A — RESULTADOS TEST")
print("=" * 55)
print(f"  Kappa cuadrático ponderado : {kappa_A:.4f}  "
      f"(baseline: {BASELINE_KAPPA:.4f}, Δ={kappa_A - BASELINE_KAPPA:+.4f})")
print(f"  F1-score macro             : {f1_A:.4f}  "
      f"(baseline: {BASELINE_F1:.4f}, Δ={f1_A - BASELINE_F1:+.4f})")
print()
print(classification_report(
    test_labels_A, test_preds_A,
    target_names=[LABEL_NAMES[i] for i in range(5)],
))

Experimento B — Radiómica (P04)

Literalmente de la Práctica 04. La novedad es usar la máscara circular del disco retiniano como ROI en lugar de una máscara de lesión, lo cual está perfectamente justificado. Útil sobre todo para demostrar interpretabilidad: qué features (entropía, contraste GLCM) distinguen los grados.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/AIM-Harvard/pyradiomics.git"],
               check=True)

import SimpleITK as sitk
import radiomics
radiomics.setVerbosity(40)
from radiomics import featureextractor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── B.1 — Configuración del extractor (igual que en P04) ─────────────────────
settings = {
    "binWidth": 25,
    "resampledPixelSpacing": None,
    "interpolator": sitk.sitkBSpline,
    "force2D": True,
}
extractor_rad = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor_rad.disableAllFeatures()
extractor_rad.enableFeatureClassByName("firstorder")
extractor_rad.enableFeatureClassByName("glcm")

print("Extractor PyRadiomics configurado.")
print("Feature classes habilitadas:", list(extractor_rad.enabledFeatures.keys()))

In [ ]:
def get_retinal_mask(img_path: Path, size: int = IMG_SIZE) -> np.ndarray:
    '''
    Genera la máscara binaria del disco retiniano (la misma región
    que delimita la máscara circular del preprocesamiento).
    Devuelve un array uint8 (0 = fondo, 1 = disco retiniano).
    '''
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    gray = img[:, :, 1]
    mask_raw = gray > 7
    rows = np.any(mask_raw, axis=1)
    cols = np.any(mask_raw, axis=0)

    if not rows.any() or not cols.any():
        return np.ones((size, size), dtype=np.uint8)

    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]
    cropped = img[r0:r1+1, c0:c1+1]

    h, w = cropped.shape[:2]
    cy, cx = h//2, w//2
    radius = min(cx, cy)
    Y, X = np.ogrid[:h, :w]
    circle = ((X-cx)**2 + (Y-cy)**2 <= radius**2).astype(np.uint8)
    circle_resized = cv2.resize(circle, (size, size), interpolation=cv2.INTER_NEAREST)
    return circle_resized


def extract_retinal_features(img_path: Path) -> dict | None:
    '''
    Extrae features radiómicos del canal verde de la retinografía
    usando la máscara del disco retiniano como ROI.
    El canal verde se usa porque es el de mayor contraste para vasos
    sanguíneos y microaneurismas en retinografías.
    '''
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    green_ch = img_rgb[:, :, 1].astype(np.float64)  # canal verde

    mask_bin = get_retinal_mask(img_path, size=IMG_SIZE)
    green_resized = cv2.resize(green_ch, (IMG_SIZE, IMG_SIZE),
                               interpolation=cv2.INTER_AREA)

    if mask_bin.sum() < 50:
        return None

    img_sitk  = sitk.GetImageFromArray(green_resized)
    mask_sitk = sitk.GetImageFromArray(mask_bin)

    try:
        result = extractor_rad.execute(img_sitk, mask_sitk)
        feats = {k: float(v) for k, v in result.items()
                 if not k.startswith("diagnostics")}
        return feats
    except Exception:
        return None


print("Funciones de extracción radiómica definidas.")

In [ ]:
# ── B.2 — Extracción de features radiómicos (con caché CSV) ───────────────────
# La extracción con PyRadiomics es lenta (~5-10 min para todo el dataset).
# Los features se guardan en CSV la primera vez; en ejecuciones posteriores
# se cargan directamente desde disco.

CACHE_TRAIN_RAD = Path("/kaggle/working/rad_features_train.csv")
CACHE_TEST_RAD  = Path("/kaggle/working/rad_features_test.csv")

if CACHE_TRAIN_RAD.exists() and CACHE_TEST_RAD.exists():
    print("[CACHE] Features radiómicos encontrados en disco — cargando...")
    df_train_rad = pd.read_csv(CACHE_TRAIN_RAD)
    df_test_rad  = pd.read_csv(CACHE_TEST_RAD)
    print(f"        Train: {len(df_train_rad)} filas | Test: {len(df_test_rad)} filas")

else:
    print("Extrayendo features radiómicos del conjunto de entrenamiento...")
    rows_train_rad = []
    for _, row in train_df.iterrows():
        path  = TRAIN_IMG / f"{row['id_code']}.png"
        feats = extract_retinal_features(path)
        if feats is not None:
            feats["diagnosis"] = int(row["diagnosis"])
            rows_train_rad.append(feats)

    print("Extrayendo features radiómicos del conjunto de test...")
    rows_test_rad = []
    for _, row in test_df.iterrows():
        path  = TEST_IMG / f"{row['id_code']}.png"
        feats = extract_retinal_features(path)
        if feats is not None:
            feats["diagnosis"] = int(row["diagnosis"])
            rows_test_rad.append(feats)

    df_train_rad = pd.DataFrame(rows_train_rad)
    df_test_rad  = pd.DataFrame(rows_test_rad)

    # Guardar en CSV para reutilización
    df_train_rad.to_csv(CACHE_TRAIN_RAD, index=False)
    df_test_rad.to_csv(CACHE_TEST_RAD,   index=False)
    print(f"Features guardados en {CACHE_TRAIN_RAD} y {CACHE_TEST_RAD}")

feat_cols_rad = [c for c in df_train_rad.columns if c != "diagnosis"]

print(f"\nFeatures extraídos por imagen : {len(feat_cols_rad)}")
print(f"Imágenes procesadas (train)   : {len(df_train_rad)}")
print(f"Imágenes procesadas (test)    : {len(df_test_rad)}")


In [ ]:
# ── B.3 — Distribución de 4 features representativos ─────────────────────────
# Igual que en P04: visualizamos si las distribuciones por clase se separan.

selected_feats = [
    "original_firstorder_Entropy",
    "original_firstorder_Mean",
    "original_glcm_Contrast",
    "original_glcm_Homogeneity",
]
selected_feats = [f for f in selected_feats if f in feat_cols_rad]

colors_grade = {
    0: "#4CAF50", 1: "#FFC107", 2: "#FF9800",
    3: "#F44336", 4: "#9C27B0",
}

fig, axes = plt.subplots(1, len(selected_feats), figsize=(5*len(selected_feats), 4))
if len(selected_feats) == 1:
    axes = [axes]

for ax, feat in zip(axes, selected_feats):
    for grade in range(5):
        vals = df_train_rad[df_train_rad["diagnosis"] == grade][feat].dropna()
        ax.hist(vals, bins=25, alpha=0.5, density=True,
                label=f"G{grade}", color=colors_grade[grade])
    ax.set_title(feat.split("_")[-1], fontsize=11, fontweight="bold")
    ax.set_xlabel("Valor del feature")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=8)

plt.suptitle("Distribución de features radiómicos por grado de RD",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── B.4 — Clasificador Logístico L1 (igual que P04) ──────────────────────────
# Penalización L1 → selección automática de features relevantes (la mayoría
# de los coeficientes quedará a cero).

X_train_rad = df_train_rad[feat_cols_rad].fillna(0).values
y_train_rad = df_train_rad["diagnosis"].values
X_test_rad  = df_test_rad[feat_cols_rad].fillna(0).values
y_test_rad  = df_test_rad["diagnosis"].values

pipe_rad = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l1", solver="saga", C=0.5,
        max_iter=5000, random_state=42,
        multi_class="multinomial",
    )),
])

# Validación cruzada 5-fold sobre train (como en P04)
cv_scores = cross_val_score(pipe_rad, X_train_rad, y_train_rad,
                             cv=5, scoring="roc_auc_ovr_weighted")
print(f"AUC OVR ponderada (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Entrenamiento final y evaluación en test
pipe_rad.fit(X_train_rad, y_train_rad)
preds_rad = pipe_rad.predict(X_test_rad)

kappa_rad = quadratic_kappa(preds_rad, y_test_rad)
f1_rad    = f1_score(y_test_rad, preds_rad, average="macro")

print()
print("=" * 55)
print("EXPERIMENTO B — RESULTADOS TEST (Radiómica)")
print("=" * 55)
print(f"  Kappa cuadrático ponderado : {kappa_rad:.4f}  "
      f"(baseline DL: {BASELINE_KAPPA:.4f})")
print(f"  F1-score macro             : {f1_rad:.4f}  "
      f"(baseline DL: {BASELINE_F1:.4f})")
print()
print(classification_report(
    y_test_rad, preds_rad,
    target_names=[LABEL_NAMES[i] for i in range(5)],
))

# Features no-nulos seleccionados por L1
coefs = np.abs(pipe_rad["clf"].coef_).mean(axis=0)
top_idx = np.argsort(coefs)[::-1][:10]
print("Top-10 features radiómicos (por |coeficiente| promedio):")
for rank, idx in enumerate(top_idx):
    if coefs[idx] > 1e-6:
        print(f"  {rank+1:2d}. {feat_cols_rad[idx]}: {coefs[idx]:.4f}")

Experimento C — BiomedCLIP zero-shot (P04)

También literalmente de P04. Lo interesante aquí es que tenemos muy pocos datos de G3 (17 en test), y un modelo zero-shot entrenado en 15M de pares PubMed podría ser sorprendentemente competitivo en esos casos. La comparación de prompts genéricos vs. clínicos detallados replica exactamente el Ejercicio 4 de P04.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"],
               check=True)

import open_clip
from PIL import Image as PILImage

print("Cargando BiomedCLIP...")
biomedclip_id = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

biomedclip_model, biomedclip_preprocess = open_clip.create_model_from_pretrained(
    biomedclip_id
)
biomedclip_tokenizer = open_clip.get_tokenizer(biomedclip_id)
biomedclip_model = biomedclip_model.to(DEVICE)
biomedclip_model.eval()

print("BiomedCLIP cargado correctamente.")

In [ ]:
# ── C.1 — Definición de prompts (genéricos vs. clínicos detallados) ───────────

PROMPTS_GENERIC = {
    0: "a retinal fundus photograph with no diabetic retinopathy",
    1: "a retinal fundus photograph with mild diabetic retinopathy",
    2: "a retinal fundus photograph with moderate diabetic retinopathy",
    3: "a retinal fundus photograph with severe diabetic retinopathy",
    4: "a retinal fundus photograph with proliferative diabetic retinopathy",
}

PROMPTS_CLINICAL = {
    0: ("a normal retinal fundus photograph showing clear optic disc, "
        "uniform background and no microaneurysms or hemorrhages"),
    1: ("a retinal fundus photograph showing mild diabetic retinopathy "
        "with a few microaneurysms and no hemorrhages or exudates"),
    2: ("a retinal fundus photograph showing moderate diabetic retinopathy "
        "with microaneurysms, dot hemorrhages, and hard exudates"),
    3: ("a retinal fundus photograph showing severe non-proliferative diabetic "
        "retinopathy with multiple blot hemorrhages in four quadrants, "
        "venous beading and intraretinal microvascular abnormalities"),
    4: ("a retinal fundus photograph showing proliferative diabetic retinopathy "
        "with neovascularization of the disc or retina and possible vitreous hemorrhage"),
}

for ptype, pdict in [("Genérico", PROMPTS_GENERIC), ("Clínico detallado", PROMPTS_CLINICAL)]:
    print(f"\n--- Prompts {ptype} ---")
    for g, txt in pdict.items():
        print(f"  G{g}: {txt[:80]}...")

In [ ]:
# ── C.2 — Función de predicción zero-shot ─────────────────────────────────────

def predict_zero_shot(img_paths: list[Path], prompts: dict, batch_size: int = 32):
    '''
    Clasifica imágenes usando similitud coseno entre embeddings visuales
    y textuales de BiomedCLIP.

    Args:
        img_paths: lista de rutas a imágenes.
        prompts  : diccionario {grado: texto} con los prompts de cada clase.
        batch_size: imágenes procesadas por lote.
    Returns:
        Array de predicciones (enteros 0-4).
    '''
    # Pre-tokenizar los textos (solo una vez)
    texts = [prompts[g] for g in range(5)]
    txt_tokens = biomedclip_tokenizer(texts, context_length=256).to(DEVICE)

    with torch.no_grad():
        _, text_feats, _ = biomedclip_model(
            torch.zeros(1, 3, 224, 224, device=DEVICE), txt_tokens
        )
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

    all_preds = []

    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i : i + batch_size]
        imgs_tensor = []

        for p in batch_paths:
            pil = PILImage.open(p).convert("RGB")
            imgs_tensor.append(biomedclip_preprocess(pil))

        imgs_tensor = torch.stack(imgs_tensor).to(DEVICE)

        with torch.no_grad():
            img_feats, _, logit_scale = biomedclip_model(imgs_tensor, txt_tokens)
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            sims      = (logit_scale * img_feats @ text_feats.T)
            preds     = sims.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds)

    return np.array(all_preds)


# Rutas de las imágenes de test (originales, sin preprocesar)
test_paths = [TEST_IMG / f"{row['id_code']}.png" for _, row in test_df.iterrows()]
test_labels_C = test_df["diagnosis"].values

print(f"Clasificando {len(test_paths)} imágenes con BiomedCLIP...")
print("(esto puede tardar 1-2 minutos)")

In [ ]:
# ── C.3 — Predicciones con ambas estrategias ──────────────────────────────────
preds_C_generic  = predict_zero_shot(test_paths, PROMPTS_GENERIC)
preds_C_clinical = predict_zero_shot(test_paths, PROMPTS_CLINICAL)

kappa_C_gen = quadratic_kappa(preds_C_generic,  test_labels_C)
kappa_C_cli = quadratic_kappa(preds_C_clinical, test_labels_C)
f1_C_gen    = f1_score(test_labels_C, preds_C_generic,  average="macro")
f1_C_cli    = f1_score(test_labels_C, preds_C_clinical, average="macro")

print("=" * 60)
print("EXPERIMENTO C — RESULTADOS TEST (BiomedCLIP zero-shot)")
print("=" * 60)
print(f"  Prompts genéricos  — Kappa: {kappa_C_gen:.4f}  F1: {f1_C_gen:.4f}")
print(f"  Prompts clínicos   — Kappa: {kappa_C_cli:.4f}  F1: {f1_C_cli:.4f}")
print(f"  Baseline DL        — Kappa: {BASELINE_KAPPA:.4f}  F1: {BASELINE_F1:.4f}")

# Mejor versión
best_kappa_C  = max(kappa_C_gen, kappa_C_cli)
best_preds_C  = preds_C_clinical if kappa_C_cli >= kappa_C_gen else preds_C_generic
best_label_C  = "clínicos" if kappa_C_cli >= kappa_C_gen else "genéricos"
print(f"\n  Mejor variante: prompts {best_label_C}")

print()
print(classification_report(
    test_labels_C, best_preds_C,
    target_names=[LABEL_NAMES[i] for i in range(5)],
))

In [ ]:
# ── C.4 — Visualización comparativa de prompts ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in [
    (axes[0], preds_C_generic,  f"Prompts genéricos (κ={kappa_C_gen:.3f})"),
    (axes[1], preds_C_clinical, f"Prompts clínicos  (κ={kappa_C_cli:.3f})"),
]:
    cm_C = confusion_matrix(test_labels_C, preds, normalize="true")
    im   = ax.imshow(cm_C, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(5))
    ax.set_xticklabels([f"G{i}" for i in range(5)])
    ax.set_yticks(range(5))
    ax.set_yticklabels([f"G{i} {LABEL_NAMES[i]}" for i in range(5)], fontsize=9)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real")
    ax.set_title(title, fontweight="bold")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{cm_C[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm_C[i,j] > 0.55 else "black", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("BiomedCLIP zero-shot — Impacto del diseño del prompt",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nConclusión Exp C:")
print(f"  Los prompts clínicos {'mejoran' if kappa_C_cli > kappa_C_gen else 'no mejoran'} "
      f"a los genéricos (Δkappa={kappa_C_cli-kappa_C_gen:+.4f}).")
print("  BiomedCLIP clasifica sin ningún entrenamiento en el dataset APTOS.")
print(f"  Diferencia frente al baseline DL: Δkappa={best_kappa_C-BASELINE_KAPPA:+.4f}")

Experimento D — Dos etapas (investigación propia, bien justificado)

Este es el que tiene mayor potencial de mejora real. El argumento es sólido: G0 es un problema trivial (casi separable linealmente) y está «distorsionando» el espacio de representación que el modelo intenta aprender para G1-G4. Separarlo en dos clasificadores independientes usa exactamente las mismas herramientas del baseline, simplemente aplicadas dos veces con datos distintos.

In [ ]:
# ── D.1 — Preparación de datasets para las dos etapas ────────────────────────

# ── Etapa 1: Binario (G0 vs G1-G4) ───────────────────────────────────────────
# Creamos una columna binaria: 0 = Sin RD, 1 = RD positivo

def make_binary_df(df):
    df2 = df.copy()
    df2["diagnosis"] = (df2["diagnosis"] > 0).astype(int)
    return df2

train_df_bin = make_binary_df(train_df)
val_df_bin   = make_binary_df(val_df)
test_df_bin  = make_binary_df(test_df)

print("Distribución binaria (Train):")
for g, name in {0: "Sin RD (G0)", 1: "RD positivo (G1-G4)"}.items():
    n = (train_df_bin["diagnosis"] == g).sum()
    pct = 100 * n / len(train_df_bin)
    print(f"  {name}: {n} ({pct:.1f}%)")

# ── Etapa 2: Severidad (solo casos G1-G4, reindexados a 0-3) ─────────────────
def make_severity_df(df):
    df2 = df[df["diagnosis"] > 0].copy()
    df2["diagnosis"] = df2["diagnosis"] - 1  # G1→0, G2→1, G3→2, G4→3
    return df2

train_df_sev = make_severity_df(train_df)
val_df_sev   = make_severity_df(val_df)
test_df_sev  = make_severity_df(test_df)

print()
print("Distribución de severidad (Train, solo RD positivos):")
sev_names = {0: "G1 Leve", 1: "G2 Moderada", 2: "G3 Severa", 3: "G4 Proliferativa"}
for g, name in sev_names.items():
    n = (train_df_sev["diagnosis"] == g).sum()
    pct = 100 * n / len(train_df_sev)
    print(f"  {name}: {n} ({pct:.1f}%)")

In [ ]:
# ── D.2 — Entrenamiento Etapa 1: Clasificador binario ────────────────────────

NUM_CLASSES_BIN = 2
NUM_EPOCHS_D    = 20

train_ds_D1 = RetinopathyDataset(
    train_df_bin, PREP_TRAIN,
    Compose([RandFlip(prob=0.5, spatial_axis=1),
             RandFlip(prob=0.5, spatial_axis=0),
             RandRotate90(prob=0.5),
             RandAffine(prob=0.3, rotate_range=0.2, scale_range=0.1,
                        padding_mode="zeros"),
             RandGaussianNoise(prob=0.2, mean=0.0, std=0.05)])
)
val_ds_D1  = RetinopathyDataset(val_df_bin,  PREP_VAL,  None)
test_ds_D1 = RetinopathyDataset(test_df_bin, PREP_TEST, None)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
train_loader_D1 = DataLoader(train_ds_D1, shuffle=True,  **loader_kw)
val_loader_D1   = DataLoader(val_ds_D1,   shuffle=False, **loader_kw)
test_loader_D1  = DataLoader(test_ds_D1,  shuffle=False, **loader_kw)

model_D1     = build_model(num_classes=NUM_CLASSES_BIN)
criterion_D1 = build_criterion(train_df_bin, n_classes=NUM_CLASSES_BIN)
optimizer_D1 = torch.optim.Adam(model_D1.parameters(), lr=LR, weight_decay=1e-5)

CKPT_D1 = Path("/kaggle/working/best_model_D1.pth")

# ── Checkpoint de caché ───────────────────────────────────────────────────────
if CKPT_D1.exists():
    print(f"[CACHE] Checkpoint encontrado: {CKPT_D1}")
    print("        Cargando pesos y omitiendo entrenamiento (Etapa 1)...")
    model_D1.load_state_dict(torch.load(CKPT_D1, map_location=DEVICE))
    model_D1.eval()
    val_preds_ck_D1, val_labels_ck_D1 = evaluate(model_D1, val_loader_D1)
    best_kappa_D1 = (val_preds_ck_D1 == val_labels_ck_D1).mean()
    print(f"        Accuracy en validación (checkpoint D1): {best_kappa_D1:.4f}")

else:
    best_kappa_D1 = -1.0

    print(f"Entrenando Etapa 1 (binario RD/no-RD) — {NUM_EPOCHS_D} épocas...")
    print("=" * 65)

    for epoch in range(1, NUM_EPOCHS_D + 1):
        train_loss_D1 = train_one_epoch_fn(model_D1, train_loader_D1,
                                            optimizer_D1, criterion_D1)
        val_preds_D1, val_labels_D1 = evaluate(model_D1, val_loader_D1)
        val_acc_D1 = (val_preds_D1 == val_labels_D1).mean()

        tag = ""
        if val_acc_D1 > best_kappa_D1:
            best_kappa_D1 = val_acc_D1
            torch.save(model_D1.state_dict(), CKPT_D1)
            tag = "  ← mejor"

        print(f"Época {epoch:02d}/{NUM_EPOCHS_D}  |  "
              f"Loss: {train_loss_D1:.4f}  |  Val acc: {val_acc_D1:.4f}{tag}")

    print(f"\nMejor accuracy validación (Etapa 1): {best_kappa_D1:.4f}")


In [ ]:
# ── D.3 — Entrenamiento Etapa 2: Clasificador de severidad ───────────────────
# SOLO con imágenes G1-G4; el modelo aprende a distinguir grados de severidad
# sin la "distracción" de la clase G0.

NUM_CLASSES_SEV = 4

# Sobremuestreo ligero para la clase G3 (Severa), la más escasa en el subconjunto
train_df_sev_os = oversample_df(train_df_sev, target_freq=0.15)
print("Distribución tras sobremuestreo (severidad):")
for g, name in sev_names.items():
    n = (train_df_sev_os["diagnosis"] == g).sum()
    pct = 100 * n / len(train_df_sev_os)
    print(f"  {name}: {n} ({pct:.1f}%)")

train_ds_D2 = RetinopathyDataset(
    train_df_sev_os, PREP_TRAIN,
    Compose([RandFlip(prob=0.5, spatial_axis=1),
             RandFlip(prob=0.5, spatial_axis=0),
             RandRotate90(prob=0.5),
             RandAffine(prob=0.3, rotate_range=0.2, scale_range=0.1,
                        padding_mode="zeros"),
             RandGaussianNoise(prob=0.2, mean=0.0, std=0.05)])
)
val_ds_D2  = RetinopathyDataset(val_df_sev,  PREP_VAL,  None)
test_ds_D2 = RetinopathyDataset(test_df_sev, PREP_TEST, None)

train_loader_D2 = DataLoader(train_ds_D2, shuffle=True,  **loader_kw)
val_loader_D2   = DataLoader(val_ds_D2,   shuffle=False, **loader_kw)
test_loader_D2  = DataLoader(test_ds_D2,  shuffle=False, **loader_kw)

model_D2     = build_model(num_classes=NUM_CLASSES_SEV)
criterion_D2 = build_criterion(train_df_sev_os, n_classes=NUM_CLASSES_SEV)
optimizer_D2 = torch.optim.Adam(model_D2.parameters(), lr=LR, weight_decay=1e-5)

CKPT_D2 = Path("/kaggle/working/best_model_D2.pth")

# ── Checkpoint de caché ───────────────────────────────────────────────────────
if CKPT_D2.exists():
    print(f"\n[CACHE] Checkpoint encontrado: {CKPT_D2}")
    print("        Cargando pesos y omitiendo entrenamiento (Etapa 2)...")
    model_D2.load_state_dict(torch.load(CKPT_D2, map_location=DEVICE))
    model_D2.eval()
    val_preds_ck_D2, val_labels_ck_D2 = evaluate(model_D2, val_loader_D2)
    best_kappa_D2 = quadratic_kappa(val_preds_ck_D2, val_labels_ck_D2)
    print(f"        Kappa en validación (checkpoint D2): {best_kappa_D2:.4f}")

else:
    best_kappa_D2 = -1.0

    print(f"\nEntrenando Etapa 2 (severidad G1-G4) — {NUM_EPOCHS_D} épocas...")
    print("=" * 65)

    for epoch in range(1, NUM_EPOCHS_D + 1):
        train_loss_D2 = train_one_epoch_fn(model_D2, train_loader_D2,
                                            optimizer_D2, criterion_D2)
        val_preds_D2, val_labels_D2 = evaluate(model_D2, val_loader_D2)
        val_kappa_D2 = quadratic_kappa(val_preds_D2, val_labels_D2)

        tag = ""
        if val_kappa_D2 > best_kappa_D2:
            best_kappa_D2 = val_kappa_D2
            torch.save(model_D2.state_dict(), CKPT_D2)
            tag = "  ← mejor"

        print(f"Época {epoch:02d}/{NUM_EPOCHS_D}  |  "
              f"Loss: {train_loss_D2:.4f}  |  Val kappa: {val_kappa_D2:.4f}{tag}")

    print(f"\nMejor kappa validación (Etapa 2): {best_kappa_D2:.4f}")


In [ ]:
# ── D.4 — Inferencia en dos etapas sobre el conjunto de test ─────────────────

model_D1.load_state_dict(torch.load(CKPT_D1, map_location=DEVICE))
model_D2.load_state_dict(torch.load(CKPT_D2, map_location=DEVICE))

@torch.no_grad()
def predict_two_stage(test_df_full, prep_dir):
    '''
    Predicción en dos etapas:
      1. Clasificador binario → ¿tiene RD? (G0 vs G1-G4)
      2. Si RD positivo → clasificador de severidad → G1/G2/G3/G4
    '''
    model_D1.eval()
    model_D2.eval()
    final_preds = []

    for _, row in test_df_full.iterrows():
        path = prep_dir / f"{row['id_code']}.png"
        img  = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        img  = img.astype(np.float32) / 255.0
        img  = (img - MEAN) / STD
        t    = torch.from_numpy(img.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        # Etapa 1: binario
        pred_bin = model_D1(t).argmax(dim=1).item()

        if pred_bin == 0:
            # Sin RD → G0
            final_preds.append(0)
        else:
            # RD positivo → Etapa 2: severidad
            pred_sev = model_D2(t).argmax(dim=1).item()
            # pred_sev está en {0,1,2,3} → {G1,G2,G3,G4}
            final_preds.append(pred_sev + 1)

    return np.array(final_preds)


print("Ejecutando inferencia en dos etapas sobre el conjunto de test...")
test_preds_D  = predict_two_stage(test_df, PREP_TEST)
test_labels_D = test_df["diagnosis"].values

kappa_D = quadratic_kappa(test_preds_D, test_labels_D)
f1_D    = f1_score(test_labels_D, test_preds_D, average="macro")

print()
print("=" * 60)
print("EXPERIMENTO D — RESULTADOS TEST (dos etapas)")
print("=" * 60)
print(f"  Kappa cuadrático ponderado : {kappa_D:.4f}  "
      f"(baseline: {BASELINE_KAPPA:.4f}, Δ={kappa_D - BASELINE_KAPPA:+.4f})")
print(f"  F1-score macro             : {f1_D:.4f}  "
      f"(baseline: {BASELINE_F1:.4f}, Δ={f1_D - BASELINE_F1:+.4f})")
print()
print(classification_report(
    test_labels_D, test_preds_D,
    target_names=[LABEL_NAMES[i] for i in range(5)],
))

RESUMEN

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────────────────────────
import pandas as pd

# Recoge los resultados de todos los experimentos (rellena los que hayas ejecutado)
resultados_finales = []

# Baseline
resultados_finales.append({
    "Experimento": "Baseline (notebook anterior)",
    "Kappa test": BASELINE_KAPPA,
    "F1 macro":   BASELINE_F1,
    "Técnica":    "EfficientNetBN + CrossEntropy ponderada",
    "Boletín":    "P04",
})

# Exp A
try:
    resultados_finales.append({
        "Experimento": "A — Sobremuestreo",
        "Kappa test": round(kappa_A, 4),
        "F1 macro":   round(f1_A, 4),
        "Técnica":    "+ Oversampling de clases minoritarias",
        "Boletín":    "P02+P04",
    })
except NameError:
    pass

# Exp B
try:
    resultados_finales.append({
        "Experimento": "B — Radiómica (LogReg L1)",
        "Kappa test": round(kappa_rad, 4),
        "F1 macro":   round(f1_rad, 4),
        "Técnica":    "PyRadiomics + Logistic L1",
        "Boletín":    "P04",
    })
except NameError:
    pass

# Exp C
try:
    resultados_finales.append({
        "Experimento": "C — BiomedCLIP zero-shot",
        "Kappa test": round(best_kappa_C, 4),
        "F1 macro":   round(max(f1_C_gen, f1_C_cli), 4),
        "Técnica":    f"BiomedCLIP prompts {best_label_C}",
        "Boletín":    "P04",
    })
except NameError:
    pass

# Exp D
try:
    resultados_finales.append({
        "Experimento": "D — Dos etapas",
        "Kappa test": round(kappa_D, 4),
        "F1 macro":   round(f1_D, 4),
        "Técnica":    "Binario RD/no-RD + Severidad G1-G4",
        "Boletín":    "Investigación propia",
    })
except NameError:
    pass

df_res = pd.DataFrame(resultados_finales).sort_values("Kappa test", ascending=False)
print("\n" + "="*65)
print("COMPARATIVA FINAL DE EXPERIMENTOS")
print("="*65)
print(df_res.to_string(index=False))

In [ ]:
# ── Gráfico de barras comparativo ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

color_map = {
    "P01": "#4CAF50", "P02+P04": "#2196F3",
    "P04": "#9C27B0", "Investigación propia": "#FF9800",
    "P03": "#F44336",
}

bar_colors = [color_map.get(row["Boletín"], "#999999")
              for _, row in df_res.iterrows()]

bars = ax.barh(
    df_res["Experimento"],
    df_res["Kappa test"],
    color=bar_colors,
    edgecolor="white",
    height=0.55,
)

# Línea de referencia del baseline
ax.axvline(BASELINE_KAPPA, color="gray", linestyle="--", linewidth=1.5,
           label=f"Baseline (κ={BASELINE_KAPPA:.4f})")

# Etiquetas de valor
for bar, val in zip(bars, df_res["Kappa test"]):
    delta = val - BASELINE_KAPPA
    delta_str = f"  κ={val:.4f} ({delta:+.4f})"
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            delta_str, va="center", fontsize=9)

ax.set_xlabel("Kappa cuadrático ponderado (test)", fontsize=11)
ax.set_title("Comparativa de experimentos — Kappa en test",
             fontsize=13, fontweight="bold")
ax.set_xlim(
    min(df_res["Kappa test"].min() - 0.1, 0.5),
    min(1.0, df_res["Kappa test"].max() + 0.08),
)

legend_patches = [
    mpatches.Patch(color=v, label=k) for k, v in color_map.items()
    if k in df_res["Boletín"].values
]
ax.legend(
    handles=legend_patches + [
        plt.Line2D([0], [0], color="gray", linestyle="--",
                   label=f"Baseline (κ={BASELINE_KAPPA:.4f})")
    ],
    loc="upper right", fontsize=9,
)

ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

<hr style="border: none; border-top: 2px solid #667eea; margin: 30px 0;">

## Experimento E — Clasificación jerárquica en tres niveles

> ⚠️ **Nota:** Este experimento extiende la investigación propia del Experimento D.
> Toda la implementación usa exclusivamente herramientas ya conocidas (MONAI,
> EfficientNetBN, PyTorch).

### Motivación: ¿por qué no funcionan bien los grados intermedios?

Los resultados de los experimentos anteriores muestran un patrón consistente:

| Grado | F1 baseline | F1 Exp D | Errores baseline |
|-------|------------|----------|-----------------|
| G0 Sin RD | 0.97 | 0.97 | 3% |
| G1 Leve | 0.51 | 0.47 | 40% |
| G2 Moderada | 0.66 | 0.66 | 39% |
| G3 Severa | 0.36 | 0.39 | 59% |
| G4 Proliferativa | 0.46 | 0.56 | 55% |

G1 y G3 son los más difíciles. Hay dos razones clínicas que lo explican:

1. **G1 (Leve) se confunde con G0 y G2**: los microaneurismas aislados de G1 son
   visualmente muy sutiles y pueden aparecer en imágenes que un experto clasificaría
   como G0 o como G2 incipiente.
2. **G3 (Severa) se confunde con G2 y G4**: la frontera entre moderada y severa (número
   de hemorragias en cuadrantes) y entre severa y proliferativa (neovascularización)
   es difusa incluso para oftalmólogos.

### Hipótesis

La diferencia visual entre **Leve+Moderada** (microlesiones pequeñas, sin
neovascularización) y **Severa+Proliferativa** (hemorragias extensas, vasos anómalos)
es mucho más pronunciada que entre grados adyacentes. Si separamos primero en estos
dos grupos, cada clasificador binario final trabajará con imágenes más homogéneas
y con más muestras por clase.

### Arquitectura propuesta: árbol jerárquico de 3 niveles

```
Imagen
  └── Etapa 1: ¿Tiene RD? (G0 vs G1-G4)          ← reutiliza model_D1
        ├── G0 → Sin Retinopatía Diabética
        └── RD positivo
              └── Etapa 2: ¿Leve-Moderada o Severa-Proliferativa?
                    ├── Grupo MILD (G1+G2)
                    │     └── Etapa 3a: ¿G1 o G2?
                    │           ├── G1 Leve
                    │           └── G2 Moderada
                    └── Grupo SEVERE (G3+G4)
                          └── Etapa 3b: ¿G3 o G4?
                                ├── G3 Severa
                                └── G4 Proliferativa
```

**Ventajas frente al Experimento D:**
- Etapa 2 resuelve una distinción visual muy clara (lesiones pequeñas vs extensas).
- Etapas 3a y 3b resuelven problemas binarios simples con imágenes más homogéneas.
- Cada clasificador tiene más muestras por clase que en el caso de 4 clases simultáneas.
- La Etapa 1 se **reutiliza directamente** del Experimento D (mismo checkpoint).


In [ ]:
# ── E.1 — Preparación de datasets para las tres etapas ───────────────────────
#
# Etapa 1 (RD / no-RD): ya entrenada en Exp D → se reutiliza CKPT_D1
#
# Etapa 2 (Mild vs Severe):
#   Mild   = G1 + G2  → etiqueta 0
#   Severe = G3 + G4  → etiqueta 1
#
# Etapa 3a (dentro de Mild):
#   G1 → etiqueta 0
#   G2 → etiqueta 1
#
# Etapa 3b (dentro de Severe):
#   G3 → etiqueta 0
#   G4 → etiqueta 1

def make_mild_severe_df(df):
    """Filtra casos RD+ y etiqueta: {G1,G2}=0 (Mild), {G3,G4}=1 (Severe)."""
    df2 = df[df["diagnosis"] > 0].copy()
    df2["diagnosis"] = (df2["diagnosis"] >= 3).astype(int)
    return df2

def make_g1g2_df(df):
    """Filtra solo G1 y G2, reindexados a 0 y 1."""
    df2 = df[df["diagnosis"].isin([1, 2])].copy()
    df2["diagnosis"] = df2["diagnosis"] - 1   # G1→0, G2→1
    return df2

def make_g3g4_df(df):
    """Filtra solo G3 y G4, reindexados a 0 y 1."""
    df2 = df[df["diagnosis"].isin([3, 4])].copy()
    df2["diagnosis"] = df2["diagnosis"] - 3   # G3→0, G4→1
    return df2

# Etapa 2
train_df_E2 = make_mild_severe_df(train_df)
val_df_E2   = make_mild_severe_df(val_df)
test_df_E2  = make_mild_severe_df(test_df)

# Etapa 3a
train_df_E3a = make_g1g2_df(train_df)
val_df_E3a   = make_g1g2_df(val_df)
test_df_E3a  = make_g1g2_df(test_df)

# Etapa 3b
train_df_E3b = make_g3g4_df(train_df)
val_df_E3b   = make_g3g4_df(val_df)
test_df_E3b  = make_g3g4_df(test_df)

# ── Resumen de distribuciones ─────────────────────────────────────────────────
print("Etapa 1 (RD/no-RD): reutiliza model_D1 — ya entrenado")
print()

print("Etapa 2 — Mild {G1,G2} vs Severe {G3,G4} (train):")
for g, name in {0: "Mild  (G1+G2)", 1: "Severe (G3+G4)"}.items():
    n = (train_df_E2["diagnosis"] == g).sum()
    print(f"  {name}: {n} ({100*n/len(train_df_E2):.1f}%)")

print()
print("Etapa 3a — G1 vs G2 (train):")
for g, name in {0: "G1 Leve", 1: "G2 Moderada"}.items():
    n = (train_df_E3a["diagnosis"] == g).sum()
    print(f"  {name}: {n} ({100*n/len(train_df_E3a):.1f}%)")

print()
print("Etapa 3b — G3 vs G4 (train):")
for g, name in {0: "G3 Severa", 1: "G4 Proliferativa"}.items():
    n = (train_df_E3b["diagnosis"] == g).sum()
    print(f"  {name}: {n} ({100*n/len(train_df_E3b):.1f}%)")

In [ ]:
# ── E.2 — Entrenamiento Etapa 2: Mild vs Severe ──────────────────────────────
# La distinción entre {G1,G2} y {G3,G4} es clínicamente la más marcada:
# el primer grupo tiene microlesiones sin neovascularización, el segundo
# presenta hemorragias extensas y posibles vasos anómalos.
# Al ser un problema binario con ~1800 muestras totales, debería converger bien.

NUM_EPOCHS_E = 20
loader_kw    = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

aug_transforms = Compose([
    RandFlip(prob=0.5, spatial_axis=1),
    RandFlip(prob=0.5, spatial_axis=0),
    RandRotate90(prob=0.5),
    RandAffine(prob=0.3, rotate_range=0.2, scale_range=0.1, padding_mode="zeros"),
    RandGaussianNoise(prob=0.2, mean=0.0, std=0.05),
])

train_ds_E2 = RetinopathyDataset(train_df_E2, PREP_TRAIN, aug_transforms)
val_ds_E2   = RetinopathyDataset(val_df_E2,   PREP_VAL,   None)

train_loader_E2 = DataLoader(train_ds_E2, shuffle=True,  **loader_kw)
val_loader_E2   = DataLoader(val_ds_E2,   shuffle=False, **loader_kw)

model_E2     = build_model(num_classes=2)
criterion_E2 = build_criterion(train_df_E2, n_classes=2)
optimizer_E2 = torch.optim.Adam(model_E2.parameters(), lr=LR, weight_decay=1e-5)

CKPT_E2 = Path("/kaggle/working/best_model_E2.pth")

if CKPT_E2.exists():
    print(f"[CACHE] Checkpoint encontrado: {CKPT_E2}")
    print("        Cargando pesos (Etapa 2 Mild/Severe)...")
    model_E2.load_state_dict(torch.load(CKPT_E2, map_location=DEVICE))
    model_E2.eval()
    vp, vl = evaluate(model_E2, val_loader_E2)
    best_kappa_E2 = (vp == vl).mean()
    print(f"        Accuracy validación (checkpoint E2): {best_kappa_E2:.4f}")
else:
    best_kappa_E2 = -1.0
    print(f"Entrenando Etapa 2 (Mild vs Severe) — {NUM_EPOCHS_E} épocas...")
    print("=" * 65)
    for epoch in range(1, NUM_EPOCHS_E + 1):
        loss_e2 = train_one_epoch_fn(model_E2, train_loader_E2, optimizer_E2, criterion_E2)
        vp_e2, vl_e2 = evaluate(model_E2, val_loader_E2)
        acc_e2 = (vp_e2 == vl_e2).mean()
        tag = ""
        if acc_e2 > best_kappa_E2:
            best_kappa_E2 = acc_e2
            torch.save(model_E2.state_dict(), CKPT_E2)
            tag = "  ← mejor"
        print(f"Época {epoch:02d}/{NUM_EPOCHS_E}  |  "
              f"Loss: {loss_e2:.4f}  |  Val acc: {acc_e2:.4f}{tag}")
    print(f"\nMejor accuracy validación (Etapa 2): {best_kappa_E2:.4f}")

In [ ]:
# ── E.3 — Entrenamiento Etapa 3a: G1 (Leve) vs G2 (Moderada) ────────────────
# Problema binario dentro del grupo Mild.
# G1 tiene microaneurismas aislados; G2 añade hemorragias y exudados.
# Al tener menos muestras (~300 G1 + ~800 G2 en train), aplicamos sobremuestreo
# de G1 para equilibrar ligeramente la distribución.

train_df_E3a_os = oversample_df(train_df_E3a, target_freq=0.35)
print("Distribución Etapa 3a tras sobremuestreo:")
for g, name in {0: "G1 Leve", 1: "G2 Moderada"}.items():
    n = (train_df_E3a_os["diagnosis"] == g).sum()
    print(f"  {name}: {n} ({100*n/len(train_df_E3a_os):.1f}%)")

train_ds_E3a = RetinopathyDataset(train_df_E3a_os, PREP_TRAIN, aug_transforms)
val_ds_E3a   = RetinopathyDataset(val_df_E3a,      PREP_VAL,   None)

train_loader_E3a = DataLoader(train_ds_E3a, shuffle=True,  **loader_kw)
val_loader_E3a   = DataLoader(val_ds_E3a,   shuffle=False, **loader_kw)

model_E3a     = build_model(num_classes=2)
criterion_E3a = build_criterion(train_df_E3a_os, n_classes=2)
optimizer_E3a = torch.optim.Adam(model_E3a.parameters(), lr=LR, weight_decay=1e-5)

CKPT_E3a = Path("/kaggle/working/best_model_E3a.pth")

if CKPT_E3a.exists():
    print(f"\n[CACHE] Checkpoint encontrado: {CKPT_E3a}")
    model_E3a.load_state_dict(torch.load(CKPT_E3a, map_location=DEVICE))
    model_E3a.eval()
    vp, vl = evaluate(model_E3a, val_loader_E3a)
    best_kappa_E3a = (vp == vl).mean()
    print(f"        Accuracy validación (checkpoint E3a): {best_kappa_E3a:.4f}")
else:
    best_kappa_E3a = -1.0
    print(f"\nEntrenando Etapa 3a (G1 vs G2) — {NUM_EPOCHS_E} épocas...")
    print("=" * 65)
    for epoch in range(1, NUM_EPOCHS_E + 1):
        loss_e3a = train_one_epoch_fn(model_E3a, train_loader_E3a, optimizer_E3a, criterion_E3a)
        vp_e3a, vl_e3a = evaluate(model_E3a, val_loader_E3a)
        acc_e3a = (vp_e3a == vl_e3a).mean()
        tag = ""
        if acc_e3a > best_kappa_E3a:
            best_kappa_E3a = acc_e3a
            torch.save(model_E3a.state_dict(), CKPT_E3a)
            tag = "  ← mejor"
        print(f"Época {epoch:02d}/{NUM_EPOCHS_E}  |  "
              f"Loss: {loss_e3a:.4f}  |  Val acc: {acc_e3a:.4f}{tag}")
    print(f"\nMejor accuracy validación (Etapa 3a): {best_kappa_E3a:.4f}")

In [ ]:
# ── E.4 — Entrenamiento Etapa 3b: G3 (Severa) vs G4 (Proliferativa) ──────────
# Problema binario dentro del grupo Severe.
# G3 tiene hemorragias extensas sin neovascularización visible;
# G4 presenta neovascularización activa y posible hemorragia vítrea.
# G3 tiene pocas muestras (~154 train), así que aplicamos sobremuestreo fuerte.

train_df_E3b_os = oversample_df(train_df_E3b, target_freq=0.35)
print("Distribución Etapa 3b tras sobremuestreo:")
for g, name in {0: "G3 Severa", 1: "G4 Proliferativa"}.items():
    n = (train_df_E3b_os["diagnosis"] == g).sum()
    print(f"  {name}: {n} ({100*n/len(train_df_E3b_os):.1f}%)")

train_ds_E3b = RetinopathyDataset(train_df_E3b_os, PREP_TRAIN, aug_transforms)
val_ds_E3b   = RetinopathyDataset(val_df_E3b,      PREP_VAL,   None)

train_loader_E3b = DataLoader(train_ds_E3b, shuffle=True,  **loader_kw)
val_loader_E3b   = DataLoader(val_ds_E3b,   shuffle=False, **loader_kw)

model_E3b     = build_model(num_classes=2)
criterion_E3b = build_criterion(train_df_E3b_os, n_classes=2)
optimizer_E3b = torch.optim.Adam(model_E3b.parameters(), lr=LR, weight_decay=1e-5)

CKPT_E3b = Path("/kaggle/working/best_model_E3b.pth")

if CKPT_E3b.exists():
    print(f"\n[CACHE] Checkpoint encontrado: {CKPT_E3b}")
    model_E3b.load_state_dict(torch.load(CKPT_E3b, map_location=DEVICE))
    model_E3b.eval()
    vp, vl = evaluate(model_E3b, val_loader_E3b)
    best_kappa_E3b = (vp == vl).mean()
    print(f"        Accuracy validación (checkpoint E3b): {best_kappa_E3b:.4f}")
else:
    best_kappa_E3b = -1.0
    print(f"\nEntrenando Etapa 3b (G3 vs G4) — {NUM_EPOCHS_E} épocas...")
    print("=" * 65)
    for epoch in range(1, NUM_EPOCHS_E + 1):
        loss_e3b = train_one_epoch_fn(model_E3b, train_loader_E3b, optimizer_E3b, criterion_E3b)
        vp_e3b, vl_e3b = evaluate(model_E3b, val_loader_E3b)
        acc_e3b = (vp_e3b == vl_e3b).mean()
        tag = ""
        if acc_e3b > best_kappa_E3b:
            best_kappa_E3b = acc_e3b
            torch.save(model_E3b.state_dict(), CKPT_E3b)
            tag = "  ← mejor"
        print(f"Época {epoch:02d}/{NUM_EPOCHS_E}  |  "
              f"Loss: {loss_e3b:.4f}  |  Val acc: {acc_e3b:.4f}{tag}")
    print(f"\nMejor accuracy validación (Etapa 3b): {best_kappa_E3b:.4f}")

In [ ]:
# ── E.5 — Inferencia jerárquica en tres niveles ──────────────────────────────
# Cargamos los mejores checkpoints de todas las etapas y ejecutamos
# el árbol de decisión completo imagen a imagen.

# Etapa 1: reutilizamos model_D1 ya cargado del Experimento D
# (si no está en memoria, lo recargamos)
if not CKPT_D1.exists():
    raise FileNotFoundError(
        "No se encontró best_model_D1.pth. Ejecuta primero el Experimento D."
    )

model_D1.load_state_dict(torch.load(CKPT_D1, map_location=DEVICE))
model_E2.load_state_dict(torch.load(CKPT_E2,  map_location=DEVICE))
model_E3a.load_state_dict(torch.load(CKPT_E3a, map_location=DEVICE))
model_E3b.load_state_dict(torch.load(CKPT_E3b, map_location=DEVICE))

for m in [model_D1, model_E2, model_E3a, model_E3b]:
    m.eval()

@torch.no_grad()
def predict_hierarchical(test_df_full, prep_dir):
    """
    Árbol jerárquico de clasificación:
      Etapa 1 → ¿RD? (G0 vs G1-G4)
      Etapa 2 → ¿Mild {G1,G2} o Severe {G3,G4}?
      Etapa 3a → ¿G1 o G2?  (si Mild)
      Etapa 3b → ¿G3 o G4?  (si Severe)
    """
    final_preds = []

    for _, row in test_df_full.iterrows():
        path = prep_dir / f"{row['id_code']}.png"
        img  = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        img  = img.astype(np.float32) / 255.0
        img  = (img - MEAN) / STD
        t    = torch.from_numpy(img.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        # Etapa 1: ¿tiene RD?
        if model_D1(t).argmax(dim=1).item() == 0:
            final_preds.append(0)   # G0 — Sin RD
            continue

        # Etapa 2: ¿Mild o Severe?
        if model_E2(t).argmax(dim=1).item() == 0:
            # Mild → Etapa 3a: ¿G1 o G2?
            pred_mild = model_E3a(t).argmax(dim=1).item()
            final_preds.append(pred_mild + 1)   # 0→G1, 1→G2
        else:
            # Severe → Etapa 3b: ¿G3 o G4?
            pred_sev = model_E3b(t).argmax(dim=1).item()
            final_preds.append(pred_sev + 3)    # 0→G3, 1→G4

    return np.array(final_preds)


print("Ejecutando inferencia jerárquica (3 niveles) sobre el conjunto de test...")
test_preds_E  = predict_hierarchical(test_df, PREP_TEST)
test_labels_E = test_df["diagnosis"].values

kappa_E = quadratic_kappa(test_preds_E, test_labels_E)
f1_E    = f1_score(test_labels_E, test_preds_E, average="macro")

print()
print("=" * 60)
print("EXPERIMENTO E — RESULTADOS TEST (jerárquico 3 niveles)")
print("=" * 60)
print(f"  Kappa cuadrático ponderado : {kappa_E:.4f}  "
      f"(baseline: {BASELINE_KAPPA:.4f}, Δ={kappa_E - BASELINE_KAPPA:+.4f})")
print(f"  F1-score macro             : {f1_E:.4f}  "
      f"(baseline: {BASELINE_F1:.4f}, Δ={f1_E - BASELINE_F1:+.4f})")
print()
print(classification_report(
    test_labels_E, test_preds_E,
    target_names=[LABEL_NAMES[i] for i in range(5)],
))

In [ ]:
# ── E.6 — Comparativa visual: Exp D vs Exp E ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (preds, labels, title) in zip(axes, [
    (test_preds_D,  test_labels_D,  f"Exp D — Dos etapas (κ={kappa_D:.3f})"),
    (test_preds_E,  test_labels_E,  f"Exp E — Jerárquico 3 niveles (κ={kappa_E:.3f})"),
]):
    cm_e = confusion_matrix(labels, preds, normalize="true")
    im   = ax.imshow(cm_e, cmap="Blues", vmin=0, vmax=1)
    cnames = [f"G{i} {LABEL_NAMES[i]}" for i in range(5)]
    ax.set_xticks(range(5)); ax.set_xticklabels(cnames, rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(5)); ax.set_yticklabels(cnames, fontsize=8)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real")
    ax.set_title(title, fontweight="bold")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{cm_e[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm_e[i,j] > 0.55 else "black", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Comparativa de pipelines jerárquicos", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Tabla comparativa G3 y G4 (los grados más difíciles) ─────────────────────
print("Comparativa en los grados más difíciles (G3 y G4):")
print(f"{'Experimento':<25} {'G3 F1':>8} {'G4 F1':>8}")
print("-" * 45)

for exp_name, preds, labels in [
    ("Baseline",      test_preds,   test_labels),
    ("A Sobremuestreo", test_preds_A, test_labels_A),
    ("D Dos etapas",  test_preds_D, test_labels_D),
    ("E Jerárquico",  test_preds_E, test_labels_E),
]:
    try:
        f1_per = f1_score(labels, preds, average=None)
        print(f"  {exp_name:<23} {f1_per[3]:>8.3f} {f1_per[4]:>8.3f}")
    except Exception:
        pass

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────────────────────────
import pandas as pd

resultados_finales = []

resultados_finales.append({
    "Experimento": "Baseline",
    "Kappa test": BASELINE_KAPPA,
    "F1 macro":   BASELINE_F1,
    "Técnica":    "EfficientNetBN + CrossEntropy ponderada",
    "Boletín":    "P04",
})

try:
    resultados_finales.append({
        "Experimento": "A — Sobremuestreo",
        "Kappa test": round(kappa_A, 4),
        "F1 macro":   round(f1_A, 4),
        "Técnica":    "+ Oversampling de clases minoritarias",
        "Boletín":    "P02+P04",
    })
except NameError:
    pass

try:
    resultados_finales.append({
        "Experimento": "B — Radiómica (LogReg L1)",
        "Kappa test": round(kappa_rad, 4),
        "F1 macro":   round(f1_rad, 4),
        "Técnica":    "PyRadiomics + Logistic L1",
        "Boletín":    "P04",
    })
except NameError:
    pass

try:
    resultados_finales.append({
        "Experimento": "C — BiomedCLIP zero-shot",
        "Kappa test": round(best_kappa_C, 4),
        "F1 macro":   round(max(f1_C_gen, f1_C_cli), 4),
        "Técnica":    f"BiomedCLIP prompts {best_label_C}",
        "Boletín":    "P04",
    })
except NameError:
    pass

try:
    resultados_finales.append({
        "Experimento": "D — Dos etapas",
        "Kappa test": round(kappa_D, 4),
        "F1 macro":   round(f1_D, 4),
        "Técnica":    "Binario RD/no-RD + Severidad G1-G4",
        "Boletín":    "Investigación propia",
    })
except NameError:
    pass

try:
    resultados_finales.append({
        "Experimento": "E — Jerárquico 3 niveles",
        "Kappa test": round(kappa_E, 4),
        "F1 macro":   round(f1_E, 4),
        "Técnica":    "G0 | Mild{G1,G2} vs Severe{G3,G4} | G1/G2 | G3/G4",
        "Boletín":    "Investigación propia",
    })
except NameError:
    pass

df_res = pd.DataFrame(resultados_finales).sort_values("Kappa test", ascending=False)
print("\n" + "="*70)
print("COMPARATIVA FINAL DE EXPERIMENTOS")
print("="*70)
print(df_res.to_string(index=False))


In [ ]:
# ── Gráfico de barras comparativo ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

color_map = {
    "P01": "#4CAF50", "P02+P04": "#2196F3",
    "P04": "#9C27B0", "Investigación propia": "#FF9800",
    "P03": "#F44336",
}

bar_colors = [color_map.get(row["Boletín"], "#999999")
              for _, row in df_res.iterrows()]

bars = ax.barh(
    df_res["Experimento"],
    df_res["Kappa test"],
    color=bar_colors,
    edgecolor="white",
    height=0.55,
)

# Línea de referencia del baseline
ax.axvline(BASELINE_KAPPA, color="gray", linestyle="--", linewidth=1.5,
           label=f"Baseline (κ={BASELINE_KAPPA:.4f})")

# Etiquetas de valor
for bar, val in zip(bars, df_res["Kappa test"]):
    delta = val - BASELINE_KAPPA
    delta_str = f"  κ={val:.4f} ({delta:+.4f})"
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            delta_str, va="center", fontsize=9)

ax.set_xlabel("Kappa cuadrático ponderado (test)", fontsize=11)
ax.set_title("Comparativa de experimentos — Kappa en test",
             fontsize=13, fontweight="bold")
ax.set_xlim(
    min(df_res["Kappa test"].min() - 0.1, 0.5),
    min(1.0, df_res["Kappa test"].max() + 0.08),
)

legend_patches = [
    mpatches.Patch(color=v, label=k) for k, v in color_map.items()
    if k in df_res["Boletín"].values
]
ax.legend(
    handles=legend_patches + [
        plt.Line2D([0], [0], color="gray", linestyle="--",
                   label=f"Baseline (κ={BASELINE_KAPPA:.4f})")
    ],
    loc="upper right", fontsize=9,
)

ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Análisis de resultados y líneas de mejora

### ¿Se resuelve el problema del desbalance?

No del todo. A lo largo de todos los experimentos, G3 (Severa) nunca supera F1=0.41
y G1 (Leve) se mueve entre 0.47 y 0.55. El sobremuestreo del Experimento A ayuda
ligeramente, y los pipelines jerárquicos de D y E también, pero la mejora es marginal.
La causa raíz es que **154 imágenes de entrenamiento para G3 es simplemente insuficiente**
independientemente de la arquitectura empleada. Existe un techo duro impuesto por los
datos, no por el modelo.

---

### Posibles líneas de mejora

**Preprocesamiento**
- Aumentación de color más agresiva (brillo, contraste, saturación aleatorios), ya que
  las retinografías varían mucho entre equipos.
- Aumentar el tamaño de entrada de 224 a 384 o 512 px: las lesiones de G1
  (microaneurismas) son muy pequeñas y pueden perderse al reducir tanto la imagen.

**Datos**
- Validación cruzada estratificada (K-fold) en lugar del split fijo actual. Con tan
  pocos datos de G3, el split puede ser desfavorable por azar. Usar K-fold aprovecharía
  mejor todas las muestras disponibles.

**Modelo**
- Probar DenseNet-121 (disponible en MONAI), diseñada específicamente para imagen
  médica, con skip connections que preservan mejor los detalles finos.
- Fine-tuning diferencial: congelar las primeras capas del backbone y entrenar solo las
  últimas con un learning rate mayor, ya que las capas tempranas ya capturan bien bordes
  y texturas desde ImageNet.

**Función de pérdida**
- Label smoothing: usar etiquetas suaves que reconozcan que los grados adyacentes son
  clínicamente similares (una imagen de G3 comparte rasgos con G2 y G4).

---

### Fuente de datos adicional: dataset de 2015

La causa principal de los bajos resultados en G3 es la falta de datos. Existe una vía
concreta para atacar este problema: el **dataset de la competición Kaggle 2015**
(*Diabetic Retinopathy Detection*), que contiene **35.126 retinografías** con las mismas
5 etiquetas, incluyendo **873 imágenes de G3** frente a las ~154 del train actual.

Está disponible directamente en Kaggle mediante "+ Add Input":

| Dataset | Descripción |
|---------|-------------|
| `c7934597/resized-2015-2019-diabetic-retinopathy-detection` | 2015 + 2019 ya redimensionados y combinados |
| `tanlikesmath/diabetic-retinopathy-resized` | Solo 2015, redimensionado |

La estrategia recomendada (usada por los participantes del top 2-3% de la competición)
es **preentrenar en 2015 y luego hacer fine-tuning en 2019**, no mezclar ambos
directamente, ya que las imágenes provienen de equipos distintos y tienen aspecto
diferente. Esta vía tiene el potencial real de super



# OJO, tambien podemos usar lo que se menciona del tema 5 de explicabilidad, pasar .pdf a claude para que lo sepa

CONCLUSIONES